# PS2 Computational Simulation

## Strategic Disclosure, AI Evaluation, and Capital Allocation

This notebook is the computational artifact for the PS2 research project.

### Research question

When firms know how an AI evaluator scores their disclosures, can a history-aware evaluator and an auction-based allocation mechanism reduce strategic disclosure while preserving informative pricing?

### Computational design

We compare three evaluator designs: No Memory, Naive History, and Capital-Purged History, under two allocation mechanisms: Uniform-Price Auction and Bookbuilding.

The simulation contains 200 issuers, 10 cohorts, 20 issuers per cohort, 2-cohort burn-in, 100 shares per issuance, 15 institutional investors, 35 retail investors, and seeds 42–46.

The notebook uses the same Python implementation as the computational artifact and does not depend on GitHub yet.

## 1. Mathematical model

The strategic disclosure model is $d_t=\theta_t+m_t$, where $\theta_t$ is latent quality and $m_t$ is strategic disclosure manipulation.

Manipulation has quadratic cost $c(m_t)=m_t^2/(2\gamma)$.

The AI evaluator produces $s_t=(1-\beta)\mu_0+\beta[a+b(d_t+\eta_t)]$.

Investor signals are $x_{it}=\theta_t+\xi_{it}$.

Realized performance is $y_t=\theta_t+\rho_t k_t+\epsilon_t$.

The capital-purged evaluator uses the allocation shock as an instrument for capital to estimate and remove the capital footprint before updating the disclosure-performance relationship.

In [ ]:
!pip -q install numpy pandas matplotlib


## 2. Upload the five canonical Python files

Upload `model.py`, `evaluator.py`, `auction.py`, `simulation.py`, and `run_experiment.py`.

This notebook intentionally does not depend on GitHub yet.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
required_files = ['model.py','evaluator.py','auction.py','simulation.py','run_experiment.py']
missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))
print('All five PS2 Python files are present.')
for f in required_files: print('✓', f)


## 3. Verify the model configuration

The canonical `model.py` defines the PS2 configuration: 200 issuers, 10 cohorts, 20 issuers per cohort, 2 burn-in cohorts, 100 shares, 15 institutional investors, 35 retail investors, and the calibration values stored in `Config`.

In [ ]:
from model import Config
cfg = Config()
cfg.validate()
print('PS2 configuration')
print('='*50)
for name in ['n_issuers','n_cohorts','issuers_per_cohort','burn_in_cohorts','shares','n_institutional','n_retail','n_investors','gamma','sigma_theta','sigma_eta','mu0','beta','b_true','sigma_epsilon','v_mean','v_sd','rho_mean','rho_sd']:
    print(f'{name}:', getattr(cfg,name))
print('\nConfiguration validation passed.')


## 4. Verify the evaluator treatments

The three evaluator modes are `no_memory`, `naive_history`, and `capital_purged`.

In [ ]:
from evaluator import MODES
print('Evaluator modes:', MODES)
assert MODES == ('no_memory','naive_history','capital_purged')
print('Evaluator configuration verified.')


## 5. Verify the auction mechanisms

The computational experiment compares Uniform-Price Auction and Bookbuilding.

In [ ]:
from simulation import MECHANISMS
print('Allocation mechanisms:', MECHANISMS)
assert MECHANISMS == ('uniform_price','bookbuilding')
print('Auction configuration verified.')


## 6. Inspect one complete simulation

We first run seed 42 with the no-memory evaluator and uniform-price auction. The sequential process is firm type → strategic disclosure → AI score → investor signals → bids → allocation → capital → realized performance → evaluator update.

In [ ]:
from simulation import run
example = run(seed=42, mode='no_memory', mechanism='uniform_price', cfg=cfg)
print('Example simulation completed.')
print('Number of observations:', len(example))
display(example.head(10))


In [ ]:
print('Columns generated by the simulation:')
for column in example.columns: print('-', column)


## 7. Inspect strategic variables

The simulation records latent quality, firm type, capital productivity, manipulation, disclosure, AI score, price, capital, and realized performance.

In [ ]:
cols=['theta','v','rho','m','disclosure','ai_signal','price','capital','performance']
display(example[cols].head(10))


## 8. Run the full PS2 experiment

The final experiment is 3 evaluator modes × 2 allocation mechanisms × seeds 42–46. Each treatment contains 10 cohorts × 20 issuers. The first two cohorts are burn-in observations for evaluator learning.

In [ ]:
import subprocess, sys
result=subprocess.run([sys.executable,'run_experiment.py'],capture_output=True,text=True)
print(result.stdout)
if result.returncode!=0:
    print(result.stderr)
    raise RuntimeError('run_experiment.py failed.')
print('\nFull PS2 experiment completed successfully.')


## 9. Load the generated results

The experiment generates run-level data, a summary table, five figures, and a fresh-run record.

In [ ]:
from pathlib import Path
import pandas as pd
results_dir=Path('results')
main_results=pd.read_csv(results_dir/'main_results.csv')
summary=pd.read_csv(results_dir/'summary_results.csv')
print('Main results:',main_results.shape)
print('Summary:',summary.shape)
display(summary)


## 10. Strategic disclosure

The primary behavioral outcome is mean manipulation $E[m]$. Lower manipulation indicates less strategic disclosure in the simulated environment.

In [ ]:
display(summary[['memory','mechanism','mean_manipulation','mean_manipulation_cost']])


## 11. Pricing informativeness

Pricing error is measured by $RMSE(P,\theta)$. Lower RMSE means simulated price is closer to latent quality.

In [ ]:
display(summary[['memory','mechanism','price_rmse']])


## 12. Capital allocation

Capital productivity is represented by $\rho$. We report $Corr(k,\rho)$ as a descriptive allocation measure.

In [ ]:
display(summary[['memory','mechanism','capital_rho_corr']])


## 13. Feedback bias

Because $y_t=\theta_t+\rho_t k_t+\epsilon_t$, realized performance partly depends on capital allocation. We report feedback-bias and first-stage diagnostics for the capital-purged evaluator.

In [ ]:
display(summary[['memory','mechanism','feedback_bias','footprint_cov_k_d_over_var_d','mean_first_stage_F']])


## 14. Retail access

The allocation mechanism also determines the share of the issuance allocated to retail investors.

In [ ]:
display(summary[['memory','mechanism','retail_allocation_share']])


## 15. Visual results

The experiment generates five figures: strategic disclosure, pricing error, allocation efficiency, feedback bias, and retail access.

In [ ]:
from IPython.display import Image, display
figure_files=['manipulation.png','price_rmse.png','allocation_efficiency.png','feedback_bias.png','retail_access.png']
for filename in figure_files:
    print('\n'+filename)
    display(Image(filename=str(results_dir/filename),width=700))


## 16. Check the burn-in design

The full run retains all observations. Headline analysis excludes the first two cohorts so evaluator memory has time to accumulate.

In [ ]:
print('Cohorts in full dataset:',sorted(main_results['cohort'].unique()))
analysis_results=main_results[main_results['cohort']>=cfg.burn_in_cohorts].copy()
print('Observations after burn-in:',len(analysis_results))
print('Burn-in cohorts:',cfg.burn_in_cohorts)


## 17. Reproducibility record

The computational artifact records random seeds, model parameters, number of issuers, cohort structure, investor composition, auction mechanisms, and evaluator modes.

In [ ]:
import json
record_path=results_dir/'fresh_run_record.json'
with open(record_path,'r') as f: record=json.load(f)
print(json.dumps(record,indent=2))


## 18. Connection to the PS2 research question

The simulation tests the chain: firm knows the evaluator → firm chooses strategic disclosure → AI evaluates disclosure → investors receive AI and private signals → allocation mechanism determines capital → capital affects realized performance → evaluator learns from historical performance → future firms respond to the updated evaluator.

The comparison changes evaluator memory/learning and the capital-allocation mechanism, then measures strategic disclosure, pricing informativeness, capital allocation, feedback bias, and retail access.

## 19. Game-theoretic interpretation

The computational model represents a repeated strategic environment with incomplete information. The firm's strategic choice is manipulation $m$, while history-aware evaluation creates a dynamic feedback channel because realized performance affects future evaluation.

## 20. Auction comparison

The scarce resource is newly issued equity capital. Firms seek capital; institutional and retail investors submit demand; the AI evaluator produces a valuation signal; and the allocation mechanism converts bids into price and allocation. The simulation compares Uniform-Price Auction and Bookbuilding under the same evaluator treatments.

## 21. Social-choice interpretation

The design affects issuers, institutional investors, retail investors, and future investors relying on AI-generated prices. We therefore report multiple dimensions separately rather than assuming one metric is sufficient: pricing informativeness, strategic disclosure, capital allocation, retail access, and feedback risk.

## 22. Mechanism-design interpretation

The mechanism-design question is: how should the evaluator and capital-allocation rule be designed so firms have less incentive to manipulate information while prices remain informative? The experiment changes evaluator learning and allocation rules and measures downstream outcomes.

## 23. Evidence status

The numerical outputs are simulation results, not empirical estimates for real financial markets. The Hugging Face artifact provides the behavioral interface, while this notebook provides computational evidence from the canonical Python implementation.

## 24. Final artifact-parity checklist

- [ ] Overleaf uses the same mathematical model.
- [ ] Overleaf uses the same evaluator treatments.
- [ ] Overleaf uses the same auction mechanisms.
- [ ] Hugging Face presents the same strategic disclosure setting.
- [ ] GitHub contains these exact Python files.
- [ ] Colab runs these exact Python files.
- [ ] Seeds are 42–46.
- [ ] Two-cohort burn-in is implemented consistently.
- [ ] Reported numerical results come from a fresh run.
- [ ] CSV results and fresh-run record are preserved.

# End of notebook

**Overleaf** → mathematical model and research argument  
**Hugging Face** → behavioral decision interface  
**Python** → canonical computational implementation  
**Colab** → runnable verification and fresh results  
**GitHub** → final reproducible source and release  
**A0 Poster** → integrated communication of question, evidence, limitations, and implications.